# Drought Agriculture Exposure Mapping
Buildling on the drought population exposure mapping exercise, we will use the same outputted SPEI data to understand the exposure and risk to crop regions. 

In [ ]:
import xarray as xr
import geemap 
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import contextily as ctx
import math 

def meters_to_decimal_degrees(meters, latitude):
    """
    Convert meters to decimal degrees based on latitude.
    
    Parameters:
    - meters (float): Distance in meters
    - latitude (float): Latitude in decimal degrees
    
    Returns:
    - float: Equivalent decimal degrees
    """
    meters_per_degree = 111320 * math.cos(math.radians(latitude))  # Adjust for latitude
    return meters / meters_per_degree

def get_spatial_bounds(file_path):
    """
    Reads GeoJSON or Shapefile and extracts the min/max latitude and longitude.

    Args:
        file_path (str): Path to the spatial file (.geojson or .shp).

    Returns:
        tuple: (min_latitude, max_latitude, min_longitude, max_longitude)
    """
    # Load the file into a GeoDataFrame
    gdf = gpd.read_file(file_path)
    if gdf.crs is None:
      raise ValueError('File does not have a CRS defined')
    if gdf.crs.to_string() != 'EPSG:4326':
      print(f'EPSG:4326 is required, but file uses {gdf.crs} instead')
      raise ValueError('Incorrect spatial reference system')
    outline_geometry=gdf.geometry
    if outline_geometry.ndim>1:
        raise ValueError(f'Make sure file contains only the needed outline'
        '(polygon) so that the variable dimension is 1. Current dimensions: '
        '{outline_geometry.ndim}')
    # min_lon, min_lat, max_lon, max_lat since EPSG:4326
    minx, miny, maxx, maxy = gdf.total_bounds

    return outline_geometry, miny, maxy, minx, maxx

### Select a model and scenario to compare to historical data

In [ ]:
aoi = 'barcelona.geojson'
model = 'mri-esm2-0'
scenario = 'ssp585'
spei_case = 'extreme'
spei_data_path = 'Barcelona/Results/'
outline_geometry, miny, maxy, minx, maxx = get_spatial_bounds(aoi)

In [ ]:
historical_filename = spei_data_path + 'STATS_SPEI6_MON9_' + spei_case + '_' + model + '_historical_RF_1951-2000_1951-2014_Barcelona_1995-2014.nc'
ssp_filename = spei_data_path + 'STATS_SPEI6_MON9_' + spei_case + '_' + model + '_'+ scenario + '_RF_1951-2000_2015-2100_Barcelona_2081-2100.nc'

### Let's actually look at one of the results

There are three variables: ```frequency```, ```max_duration```, and ```max_intensity```. We can calculate the difference between the historical files and ssp scenarios, and plot any of the variables. 

Now let's select one of these variables and plot what the difference is - note that depending on the region and variable, your colormap range (i.e. ```vmin``` and ```vmax```) will have to be adjusted. 

In [ ]:
variable = 'frequency'
# Create the figure
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})


# Set the extent (bounding box of netcdf data plus some buffer)
buffer = 0.25 
lon_min, lon_max = df_historical.lon.min().item(), df_historical.lon.max().item()
lat_min, lat_max = df_historical.lat.min().item(), df_historical.lat.max().item()


ax.set_extent([lon_min-buffer, lon_max+buffer, lat_min-buffer, lat_max+buffer], crs=ccrs.PlateCarree())

# Define colormap range
vmin, vmax = -5, 5

# Plot the average difference for the selected month
pcm = difference[variable].plot(ax=ax, cmap="RdBu_r", alpha=0.7, vmin=vmin, vmax=vmax, add_colorbar=False)

# Add a title
#ax.set_title(f'SPEI{spei_dur} Month {spei_month}: {gcm.upper()} {scenario.upper()} ({start_year}-{end_year})', fontsize=16)

# Add coastlines and basemap
ax.add_feature(cfeature.COASTLINE)
ctx.add_basemap(ax, crs=gpd.read_file(aoi).crs.to_string(),
                source=ctx.providers.OpenStreetMap.Mapnik, attribution_size=6, alpha=1)

# Add gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="gray", alpha=0.5, linestyle="--")
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {"size": 12}
gl.ylabel_style = {"size": 12}

# Add colorbar
cbar = fig.colorbar(pcm, ax=ax, location="bottom", shrink=0.7, label="difference ssp-historical")
cbar.ax.tick_params(labelsize=12)
ax.set_title(variable, fontsize=16, fontweight="bold")


plt.show()

## Introducing geemap and agricultural areas to the analysis 
Google Earth Engine (GEE) is a very powerful tool that hosts many of the most popular datasets in the Google Cloud environment, and allows users to not just access the data, but to also use Google compute to run analysis. For the rest of this assignment and class, you will need to register for a GEE account: [https://code.earthengine.google.com/register](https://code.earthengine.google.com/register)

To see what datasets exist in GEE you can visit: [https://developers.google.com/earth-engine/datasets](https://developers.google.com/earth-engine/datasets)


GEE was originally offered as a free service to researchers, but you had to know how to code in Javascript. An amazing Professor at University of Tennessee Dr. Qiusheng Wu - has created a python package that allows us to access the data and leverage Google servers via python scripts. I recommend reading an introduction to this package here and look at all the amazing tutorials: [https://book.geemap.org/chapters/01_introduction.html](https://book.geemap.org/chapters/01_introduction.html)



### Practice loading in data from the GEE catalog
In the GEE catalog, the key pieces of info to pay attention to is the collection name (in this case let's use ```"MODIS/061/MCD12Q1"```, information can be found at [https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MCD12Q1](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MCD12Q1)). Take a look at the description of the bands... how can we use this dataset to look at crops?

In [ ]:
import geemap
import ee

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize()

# Define the bounding box for Catalonia, Spain
gee_bbox = ee.Geometry.Rectangle([lon_min-buffer, lat_min-buffer, lon_max+buffer, lat_max+buffer])  # Approximate bbox

# Load MODIS Land Cover dataset and select the latest year
modis = ee.ImageCollection("MODIS/061/MCD12Q1").select("LC_Type1")
latest_year = modis.sort("system:time_start", False).first()

# Define the cropland class (12 in MCD12Q1 legend) and clip to Catalonia
cropland = latest_year.eq(12).clip(gee_bbox)

# Define visualization parameters
cropland_vis = {
    "min": 0,
    "max": 1,
    "palette": ["black", "yellow"],  # yellow for cropland
}

# Create interactive map
m = geemap.Map(center=[lat_min, lon_min], zoom=7)

# Add satellite basemap for better visualization
m.add_basemap("SATELLITE")

# Add cropland layer
m.addLayer(cropland.selfMask(), cropland_vis, "Cropland in Barcelona")

# Add a red bounding box for Catalonia
m.addLayer(gee_bbox, {"color": "red"}, "Barcelona Bounding Box")



# Display the map
m


You might ask - where is this data? Well it's being streamed from the Google servers to this notebook. If we want to actually work with this crop mask and use it to derive information on exposure under drought scenarios, we can convert the GEE subset data to an xarray dataset.

In [ ]:
# Find the native resolution of the dataset (meters)
data_resolution = modis.first().projection().nominalScale().getInfo()
print(f"Data Resolution: {data_resolution} meters")

# convert meters to a rough decimal degree

data_resolution_degrees = meters_to_decimal_degrees(data_resolution,lat_max)
print(f"Data Resolution: {data_resolution_degrees} degrees")


In [ ]:
crop_ds = geemap.ee_to_xarray(
    dataset = cropland,
    geometry=gee_bbox,
    scale=data_resolution_degrees,  # Preserve the original dataset resolution
    crs="EPSG:4326"  # Ensure the correct CRS
)

In [ ]:
#crop_ds = crop_ds1.transpose("time", "lat", "lon")
crop_ds['LC_Type1'] = crop_ds['LC_Type1'].transpose("time", "lat", "lon")
crop_ds

In [ ]:
crop_ds['LC_Type1'][0].plot()

In this region - how much can we estimate is croplands? This is raster data, so we can count the number of '1's (where 1 = crop and 0 = no crop) and then multiply by the area of each pixel. We already calculated the original resolution 

In [ ]:
# Compute pixel area in km²
pixel_area_km2 = (data_resolution * data_resolution) / 1e6  # Convert m² to km²

# Count crop pixels (where mask = 1)
crop_pixel_count = crop_ds['LC_Type1'][0].sum().item()

#  Estimate total crop area
total_crop_area_km2 = crop_pixel_count * pixel_area_km2
print(f"Total crop area in region: {total_crop_area_km2} km2")


### Exposure of crop areas to changes in drought frequency 
Earlier we compared the difference between historical SPEI and ssp5 future scenarios of SPEI. For whatever variable you chose to compare, you can come up with some informed thresholds to create a cross mask and look at the exposure of crops to changes in drought intervals in the future. 

For this example I will look at changes in frequency and set a threshold with a logic:
**tell me the area of crops where droughts will occur >3x more frequently than historic levels**

In [ ]:
threshold = 2
change_frequency = difference['frequency']/df_historical['frequency']
masked_ds = (change_frequency >= threshold).astype(int)
masked_ds

This masked file I now want to interpolate to the same resolution as the crop mask which is 0.005578 degrees. The original SPEI data is at 0.5 degrees. xarray has a function to do this: 

In [ ]:
downscaled_masked_ds = masked_ds.interp(
    lat=crop_ds.lat, 
    lon=crop_ds.lon, 
    method="nearest"  # Other options: "nearest", "cubic"
)

In [ ]:
downscaled_masked_ds

In [ ]:
downscaled_masked_ds.plot()

Now we can multiply the two masks

In [ ]:
cross_mask=crop_ds['LC_Type1'][0] * downscaled_masked_ds

In [ ]:
cross_mask.plot()

In [ ]:
#  Estimate total crop area susceptible to more frequent droughts
# Count crop pixels (where mask = 1)
crop_pixel_count = cross_mask.sum().item()

total_crop_area_km2 = crop_pixel_count * pixel_area_km2
print(f"Total crop area in region: {total_crop_area_km2} km2")

## Conclusions

Now you have a way to map more exposed agriculture regions and make a statement of the area expected to be impacted. 